# Pipeline - Modo Nao-Estruturado
Avalia um modelo LLM em um dataset de multipla escolha.
O modelo responde livremente e um **LLM Judge** extrai a letra da resposta.

## Configuracao
Altere as variaveis abaixo antes de executar.

In [ ]:
# Modelo avaliado
MODEL          = 'phi'          # ex: phi, llama3.1:latest, gpt-4o-mini
PROVIDER       = 'ollama'       # ollama | openai
TEMPERATURE    = 0.0

# Modelo juiz (extrai a letra da resposta livre)
JUDGE          = 'gpt-4o-mini'  # ex: gpt-4o-mini, llama3.1:latest
JUDGE_PROVIDER = 'openai'       # ollama | openai

# Dataset
# Aliases: agievalar-dev/test, aqua-train/test/validation,
#          arc-train/test/validation, gsm8k-train/test,
#          mmlu-test/validation
# Ou caminho direto para um .parquet
DATASET        = 'agievalar-test"'

# Tamanho da amostra (None = dataset inteiro)
QUESTION_LIMIT = 50

# Saida
OUTPUT_DIR     = 'results'
DEBUG          = False


In [ ]:
BUILTIN_DATASETS = {
    "agievalar-dev": "datasets/agievalar/dev.parquet",
    "agievalar-test": "datasets/agievalar/test.parquet",
    "aqua-train": "datasets/aqua/train.parquet",
    "aqua-test": "datasets/aqua/test.parquet",
    "aqua-validation": "datasets/aqua/validation.parquet",
    "arc-train": "datasets/arc/arc_train.parquet",
    "arc-test": "datasets/arc/arc_test.parquet",
    "arc-validation": "datasets/arc/arc_validation.parquet",
    "gsm8k-train": "datasets/gsm8k/train.parquet",
    "gsm8k-test": "datasets/gsm8k/test.parquet",
    "mmlu-test": "datasets/mmlu/test.parquet",
    "mmlu-validation": "datasets/mmlu/validation.parquet",
}


## Imports e path

In [ ]:
import sys
from pathlib import Path

ROOT = Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv()

from src.models import load_model
from src.pipeline import BUILTIN_DATASETS, load_dataset, run_dataset, save_results

print('ROOT:', ROOT)
print('Datasets disponíveis:', list(BUILTIN_DATASETS.keys()))


## Carregamento dos modelos

In [ ]:
model = load_model(MODEL, PROVIDER, TEMPERATURE)
judge = load_model(JUDGE, JUDGE_PROVIDER, temperature=0.0)

print(f'Modelo : {MODEL} [{PROVIDER}]')
print(f'Juiz   : {JUDGE} [{JUDGE_PROVIDER}]')


## Carregamento do dataset

In [ ]:
dataset = load_dataset(DATASET, samples=QUESTION_LIMIT)

print(f'Dataset  : {DATASET}')
print(f'Amostras : {len(dataset)}')
print(f'Colunas  : {dataset.column_names}')
print('
Exemplo (item 0):')
print(dataset[0])


## Execucao do pipeline (nao-estruturado)

In [ ]:
results = run_dataset(
    model=model,
    dataset=dataset,
    judge=judge,
    structured=False,
    debug=DEBUG,
    generate_critique_flag=False,
)

total   = len(results)
correct = sum(v['flag_acerto'] for v in results.values())
acc     = correct / total if total else 0.0

print(f'
{"=" * 40}')
print(f'  Total    : {total}')
print(f'  Corretos : {correct}')
print(f'  Acuracia : {acc:.1%}')
print(f'{"=" * 40}')


## Salvar resultados

In [ ]:
dataset_label = DATASET.replace('/', '_').replace('\', '_').replace('.parquet', '')

summary, json_file, csv_file = save_results(
    results,
    output_path=OUTPUT_DIR,
    model_name=MODEL,
    dataset_name=dataset_label,
)

print('JSON:', json_file)
print('CSV :', csv_file)


## Analise dos resultados

In [ ]:
import pandas as pd

rows = []
for idx, row in results.items():
    rows.append({
        '#':            idx,
        'Correct':      row['correct_answer_alt'],
        'Predicted':    row['predicted_alt'],
        'Hit':          'OK' if row['flag_acerto'] else 'WRONG',
        'Correct Text': row['correct_answer_text'],
    })

df = pd.DataFrame(rows)

wrong_df = df[df['Hit'] == 'WRONG']
print(f'Erros: {len(wrong_df)}/{total}')
display(wrong_df.head(20))


## Visualizacoes

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(
    ['Corretas', 'Erradas'],
    [correct, total - correct],
    color=['#4C72B0', '#DD8452'],
    edgecolor='white',
)
axes[0].set_title(f'{MODEL} - {DATASET}', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Questoes')
axes[0].text(0, correct + 0.5, f'{acc:.1%}', ha='center', fontsize=12, color='#333')
axes[0].spines[['top', 'right']].set_visible(False)

pred_counts = df['Predicted'].value_counts().sort_index()
axes[1].bar(pred_counts.index, pred_counts.values, color='#55A868', edgecolor='white')
axes[1].set_title('Distribuicao das respostas previstas', fontsize=12)
axes[1].set_xlabel('Label previsto')
axes[1].set_ylabel('Frequencia')
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
safe_model = MODEL.replace(':', '_')
out_path = Path(OUTPUT_DIR) / f'{safe_model}_{dataset_label}_analysis.png'
plt.savefig(out_path, dpi=150)
print('Plot salvo em:', out_path)
plt.show()
